# 03 — Video Preprocessing

Design a reproducible, resumable video feature pipeline using a fixed number of sampled frames and a compact image size.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

In [ ]:
from pathlib import Path
import json, random
import numpy as np

SEED = 42
NUM_FRAMES = 16
FRAME_HEIGHT = 96
FRAME_WIDTH = 96
CACHE_DIR = Path("../../data/processed/video_frames")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache directory:", CACHE_DIR.resolve())

## Sampling strategy
Sample 16 approximately uniform timestamps per clip. Decode with OpenCV, convert BGR to RGB, resize to 96×96, normalize to `[0, 1]`, and save each sample with its stable record ID. Keep a manifest linking cached features to source files and labels.

In [ ]:
def uniform_indices(frame_count: int, num_frames: int = NUM_FRAMES) -> np.ndarray:
    """Return sorted frame indices spanning the available clip."""
    if frame_count <= 0:
        return np.array([], dtype=np.int64)
    return np.linspace(0, frame_count - 1, num=min(frame_count, num_frames), dtype=np.int64)

print("Example indices for 120-frame clip:", uniform_indices(120))

## Decoder implementation
Implement and validate decoding on a small sample before processing the full dataset. Catch unreadable videos, log them to a failure manifest, and never silently relabel failed samples. Store outputs in chunks and skip IDs already present in the manifest.

In [ ]:
try:
    import cv2
    print("OpenCV:", cv2.__version__)
except ImportError:
    print("Install opencv-python before running video decoding.")

### Validation checklist
- Confirm RGB channel order and output shape `(T, 96, 96, 3)`.
- Inspect sample frames visually.
- Record failed decodes and actual frame counts.
- Never overwrite original media.